# 문장 유형화 EDA (구조 수치 기반)

**목적**: gemma 라벨(train 전량 9,535)의 **개수 축**(n_events/n_subj/n_markers)과 불리언 축(camera/viewer)으로
해석 가능한 유형 체계를 만들고, holdout 실측으로 약점 유형을 확정한다.
문장 추출물 자체(events 텍스트)는 학습(v7 CoT 타깃)에서 쓰고, 여기서는 **수치만** 다룬다.

- 전 셀 **CPU 전용** — GPU 작업(라벨링·학습)과 동시 실행 가능
- 유형 함수는 `structure_features.assign_types()` — train/test 단일 진실. 컷포인트 확정 시 기본값 갱신
- ⚠️ 규정: 약점 유형 → 증강 가중의 근거는 **holdout 실측만** 문서화 (test 분포 인용 = 실격, PROJECT_SETUP §4.3)

In [ ]:
# ① 전량 피처 테이블 (train.csv 9,535 전체 — holdout 포함, is_holdout 플래그)
import sys; sys.path.insert(0, "scripts")
import pandas as pd
from structure_features import build_feature_table, assign_types

full = pd.read_csv("snuaichallenge_data/train.csv")
ft = build_feature_table(full)                     # camera_re + gemma 축 + CLIP 축 병합
hold_ids = set(pd.read_csv("splits/holdout_300.csv").Id)
ft["is_holdout"] = ft.Id.isin(hold_ids)
print(f"gemma 결측 {ft.n_events.isna().sum()}행 (0이어야 전량 커버)")
ft.head(3)

In [ ]:
# ② 축별 정확도 프로파일 (holdout x exp07, 섞인 샘플만) — 컷포인트 결정의 근거
#    ⚠️ 원본 n_events/n_subj는 camO에서 카메라 몫만큼 부풀어 camera 축과 교란됨 (7/18 실측:
#    camO의 80.8%가 카메라 사건 포함, 평균 +1.1개) — 유형 축은 *_noncam 사용이 기본
pred = pd.read_csv("./outputs/preds/Qwen3-VL-2B-Instruct_exp07_aug2_full.csv")
pred = pred[pred["gt"] != "[1, 2, 3, 4]"]
h = pred.merge(ft, on="Id", how="left")
print(f"검증 표본: 섞인 holdout {len(h)}개\n")
for col in ["n_events", "n_events_noncam", "n_subj", "n_subj_noncam", "n_markers", "camera", "viewer"]:
    t = h.dropna(subset=[col]).groupby(col).correct.agg(["mean", "size"])
    print(f"[{col}]")
    print((t["mean"].map("{:.1%}".format) + "  n=" + t["size"].astype(str)).to_string(), "\n")

In [ ]:
# ③ 컷포인트 설정 (② 프로파일 보고 숫자만 수정) -> 유형 부여 + 유형별 검증
EVENTS_CUTS = (2,)               # (2,) = 비카메라 사건 <=2 sparse / >=3 dense (4유형)
DENSITY_COL = "n_events_noncam"  # 원본으로 되돌리려면 "n_events" + EVENTS_CUTS=(2, 4)

ft_t = assign_types(ft, EVENTS_CUTS, DENSITY_COL)
print("train 전량 유형 분포:")
print(ft_t.stype.value_counts().to_string(), "\n")

ht = pred.merge(ft_t, on="Id", how="left")
t = ht.groupby("stype").correct.agg(["mean", "size"]).sort_values("mean")
print("유형별 holdout 정확도 (오름차순 = 약점부터):")
print((t["mean"].map("{:.1%}".format) + "  n=" + t["size"].astype(str)).to_string())

print("\n보조 태그별:")
for tag in ["tag_multi_subj", "tag_no_marker", "tag_viewer"]:
    tt = ht.groupby(tag).correct.agg(["mean", "size"])
    print(f"  {tag}: " + " | ".join(f"{k}: {v['mean']:.1%}(n={int(v['size'])})" for k, v in tt.iterrows()))

In [ ]:
# ③b 두 기준 비교 — 원본 6유형(교란 포함) vs 비카메라 4유형(개정)
#    판단 잣대: 셀 최소 n(통계 유효성), 분리 폭, 그리고 행이 어떻게 이동했는가
v1 = assign_types(ft, (2, 4), "n_events")          # 이전: 원본 카운트 3구간 x camera
v2 = assign_types(ft, (2,), "n_events_noncam")     # 개정: 비카메라 2구간 x camera

for name, ft_v in [("v1 원본 6유형", v1), ("v2 비카메라 4유형", v2)]:
    hv = pred.merge(ft_v, on="Id", how="left")
    t = hv.groupby("stype").correct.agg(["mean", "size"]).sort_values("mean")
    spread = t["mean"].max() - t["mean"].min()
    print(f"== {name} == (분리 폭 {spread:.1%}, 최소 셀 n={int(t['size'].min())})")
    print((t["mean"].map("{:.1%}".format) + "  n=" + t["size"].astype(str)).to_string(), "\n")

mig = pd.crosstab(v1.stype, v2.stype)
print("행 이동 (행=v1, 열=v2) — camO 행이 어디로 재배치됐는지가 교란 제거의 핵심:")
print(mig.to_string())

In [ ]:
# ④ 독립성 교차 — 기존 축(spacy Partition, CLIP scene_cuts)과 겹치는 정보인지
team = pd.read_csv("train_검토_최종_완료.csv", encoding="cp949")[["Id", "Partition"]]
x = ht.merge(team, on="Id", how="left")
print("stype x Partition (행 비율) — 대각 집중이면 중복 정보, 분산이면 독립:")
print(pd.crosstab(x.stype, x.Partition, normalize="index").round(2).to_string(), "\n")
print("stype x scene_cuts별 정확도 (n>=10 셀만):")
t = x.groupby(["stype", "scene_cuts"]).correct.agg(["mean", "size"])
print((t[t["size"] >= 10]["mean"].map("{:.1%}".format) + "  n=" + t[t["size"] >= 10]["size"].astype(str)).to_string())

In [ ]:
# ⑤ 저장 — train_types.csv (+ test는 추출 완료 시 같은 함수로)
import os
cols = ["Id", "camera", "viewer", "n_events", "n_events_noncam", "n_subj", "n_subj_noncam",
        "n_markers", "scene_cuts", "n_similar", "stype", "tag_multi_subj", "tag_no_marker",
        "tag_viewer", "is_holdout"]
ft_t[cols].to_csv("./outputs/gemma_labels/train_types.csv", index=False)
print("저장: outputs/gemma_labels/train_types.csv", len(ft_t), "행")

tf_path = "./outputs/gemma_labels/test_features.csv"
if os.path.exists(tf_path):
    tf = pd.read_csv(tf_path)
    if len(tf) >= 819:      # 추출 완료분만
        tt = assign_types(tf, EVENTS_CUTS, DENSITY_COL)
        tt.to_csv("./outputs/gemma_labels/test_types.csv", index=False)
        print("저장: test_types.csv", len(tt), "행 (추론 라우팅·제출 후 분석 전용 — 학습 설계 사용 금지)")
    else:
        print(f"test 추출 진행 중 ({len(tf)}/819) — 완료 후 이 셀 재실행")

In [ ]:
# ⑥ 증강 RULES 후보 — 유형x태그 조합의 약점 순위 (holdout 근거 문서화용)
combo = ht.groupby(["stype", "tag_multi_subj"]).correct.agg(["mean", "size"])
combo = combo[combo["size"] >= 12].sort_values("mean")
print("약점 조합 순위 (n>=12, holdout 실측 — 증강 가중 근거는 이 표만 인용할 것):")
print((combo["mean"].map("{:.1%}".format) + "  n=" + combo["size"].astype(str)).to_string())
print("\n-> Mini_Hint_Experiments.ipynb ⑩ RULES에 이 조건을 옮겨 CSV 생성")